# 多 Provider 切换

`nova_ai` 通过统一的 `Model` 对象屏蔽不同厂商差异，只需修改 `get_model` 的 provider 和 model id。

本 Notebook 同时列出已注册模型，并演示对同一问题使用不同厂商模型。


In [ ]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"
# os.environ["OPENAI_API_KEY"] = ""  # 本地未设置 OPENAI_API_KEY，已注释


In [ ]:
from nova_ai import list_providers, list_all_models

print("已注册 providers:", list_providers())
print("\n前 10 个模型:")
count = 0
for provider, provider_models in list_all_models().items():
    for model_id in provider_models:
        print(f"  - {provider.value}/{model_id}")
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break

## 统一调用接口

把流式消费逻辑封装成 `chat()` 函数，即可无缝切换模型。


In [ ]:
from nova_ai import get_model, UserMessage, Context, stream_simple

async def chat(model_id: str, provider: str, user_text: str):
    model = get_model(provider, model_id)
    context = Context(
        system_prompt="你是严谨的技术助手。",
        messages=[UserMessage(role="user", content=user_text)],
    )
    stream = stream_simple(model, context)
    print(f"\n>>> {provider}/{model_id}")
    async for event in stream:
        if event.type == "text_delta":
            print(event.delta, end="", flush=True)
        elif event.type == "error":
            print(f"\n[错误] {event.error}", end="")
    print("\n")

# 示例：分别调用两个厂商的模型（请确保对应 Key 已设置）
await chat("deepseek-v3-2-251201", "volcengine", "Python 的 asyncio 是什么？")
# await chat("gpt-4o", "openai", "Python 的 asyncio 是什么？")
